# Phase 1 — Notebook 1.5: Semantic Enrichment

Builds two injection maps from the preprocessed vocab, then writes an enriched parquet
that `02_tfidf_analysis.ipynb` can load in place of `05_filtered.parquet`.

| Pass | What | Output |
|------|------|--------|
| A | Subject clustering (SentenceTransformer + agglomerative) | `semantic_map.csv` |
| B | Framing taxonomy discovery + classification (LLM) | `framing_map.csv` |
| C | Token injection | `05_enriched.parquet` |

**Run order:** `01_preprocess` → `01.5_semantic_enrichment` → `02_tfidf_analysis`

**Key outputs (all human-reviewable before injection runs):**
- `OUTPUTS/enrichment/semantic_map.csv` — term → primary_category, subcategory
- `OUTPUTS/enrichment/framing_taxonomy.json` — discovered framing categories (edit before classifying)
- `OUTPUTS/enrichment/framing_map.csv` — term → framing_category
- `OUTPUTS/enrichment/injection_preview.csv` — full preview of what gets added to each unique token set

## Setup

In [1]:
import sys, os, json, time, httpx
from pathlib import Path
import pandas as pd
import numpy as np
from openai import OpenAI

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import load_cfg, flat_freq

def OUT(subdir, fname):
    p = ROOT / "OUTPUTS" / subdir / fname
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

CFG = load_cfg()

# ── OpenAI client (same pattern as 02_tfidf_analysis) ────────────────────────
if "client" not in globals():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Missing OPENAI_API_KEY environment variable.")
    http_client = httpx.Client(verify=False)
    client = OpenAI(api_key=api_key, http_client=http_client)
    print("✅ OpenAI client initialized.")
else:
    print("🔁 Client already loaded; skipping setup.")

MODEL_DISCOVERY = "gpt-5.4"        # taxonomy discovery — needs reasoning
MODEL_CLASSIFY  = "gpt-5.4-mini"   # batch classification — cheap + fast
MODEL_GATE      = "gpt-5.4-mini"   # cluster coherence gate

# ── Load preprocessed data ────────────────────────────────────────────────────
df       = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_filtered.parquet")
doc_freq = pd.read_csv(ROOT / "OUTPUTS/vocab/unigram_docfreq.csv",
                       index_col=0).squeeze().rename("doc_freq")
freq     = flat_freq(df).rename("corpus_freq")
vocab_df = pd.DataFrame({"corpus_freq": freq, "doc_freq": doc_freq}).dropna()
vocab_df["docs_pct"] = vocab_df["doc_freq"] / len(df)

print(f"Corpus:        {len(df):,} projects")
print(f"Full vocab:    {len(vocab_df):,} terms")
print(f"Rare (<1k df): {(vocab_df.doc_freq < 1000).sum():,} terms")

✅ OpenAI client initialized.
Corpus:        896,277 projects
Full vocab:    7,297 terms
Rare (<1k df): 3,622 terms


---
## Pass A — Subject clustering

Embeds rare vocab terms with SentenceTransformer, clusters with agglomerative cosine distance,
then runs each cluster through an LLM coherence gate. Only `inject` clusters become tokens.

In [2]:
# ── Parameters ────────────────────────────────────────────────────────────────
RARE_DOC_FREQ_CEILING = 1000   # only cluster terms below this doc_freq
AGG_DISTANCE_THRESHOLD = 0.35  # cosine distance cutoff (lower = tighter clusters)
MIN_CLUSTER_SIZE = 3           # discard clusters smaller than this
MIN_INJECT_PROJECTS = 25       # injected category token must reach this doc_freq to survive filter

In [3]:
import os
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

import httpx
_original_init = httpx.Client.__init__
def _patched_init(self, *args, **kwargs):
    kwargs["verify"] = False
    _original_init(self, *args, **kwargs)
httpx.Client.__init__ = _patched_init

import urllib3
urllib3.disable_warnings()

# ── params ────────────────────────────────────────────────────────────────────
RARE_DOC_FREQ_CEILING  = 1000   # same cutoff as before
AGG_DISTANCE_THRESHOLD = 0.35   # lower = tighter/more clusters; tune this
MIN_CLUSTER_SIZE       = 3      # drop singletons and pairs

# ── load vocab ────────────────────────────────────────────────────────────────
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

doc_freq = pd.read_csv("OUTPUTS/vocab/unigram_docfreq.csv", index_col=0).squeeze()
vocab_df = doc_freq.rename("doc_freq").to_frame()

# ── embed + cluster ───────────────────────────────────────────────────────────
rare_terms = vocab_df[vocab_df.doc_freq < RARE_DOC_FREQ_CEILING].index.tolist()
print(f"Embedding {len(rare_terms):,} rare terms...")

embedder   = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(rare_terms, show_progress_bar=True,
                             normalize_embeddings=True)

agg = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=AGG_DISTANCE_THRESHOLD,
    metric="cosine",
    linkage="average"
)
labels = agg.fit_predict(embeddings)

sizes      = Counter(labels)
valid_ids  = {cid for cid, n in sizes.items() if n >= MIN_CLUSTER_SIZE}
cluster_df = pd.DataFrame({"term": rare_terms, "cluster_id": labels})
cluster_df = cluster_df[cluster_df.cluster_id.isin(valid_ids)].copy()

clusters = (cluster_df.groupby("cluster_id")["term"]
            .apply(list).reset_index(name="terms"))

print(f"\nTotal terms in valid clusters: {len(cluster_df):,} of {len(rare_terms):,}")
print(f"Valid clusters ({MIN_CLUSTER_SIZE}+ members): {len(clusters)}")
print("\nSample clusters:")
for _, row in clusters.sample(min(8, len(clusters)), random_state=42).iterrows():
    print(f"  C{row.cluster_id:03d}: {row.terms[:12]}")

Embedding 3,622 rare terms...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/114 [00:00<?, ?it/s]


Total terms in valid clusters: 294 of 3,622
Valid clusters (3+ members): 90

Sample clusters:
  C148: ['freeze', 'freezer', 'freezing']
  C075: ['corporate', 'corporation', 'incorporation']
  C220: ['gene', 'genetic', 'genetics']
  C283: ['el', 'elapse', 'elar', 'elate', 'eld']
  C004: ['guatemala', 'honduras', 'venezuela']
  C081: ['bilingualism', 'lingual', 'monolingual']
  C140: ['battleship', 'naval', 'navy']
  C267: ['dodge', 'dodgeball', 'dodgeballs']


In [4]:
# ── LLM coherence gate ────────────────────────────────────────────────────────
# One call per cluster. Returns inject / split / discard + proposed labels.

GATE_SYSTEM = (
    "You are an NLP analyst reviewing term clusters from DonorsChoose teacher project "
    "essays (K-12 US public school classrooms). These clusters come from rare vocabulary "
    "(terms appearing in fewer than 1,000 of 900,000 projects). "
    "Respond ONLY with valid JSON, no preamble, no markdown fences."
)

GATE_PROMPT = """Cluster {cid} terms: {terms}

Assess whether these terms share a coherent educational domain concept.

Return a JSON object with exactly these keys:
  action           : "inject" | "split" | "discard"
  primary_category : snake_case domain label (e.g. "marine_biology") or null
  subcategory      : more specific snake_case label or null
  split_into       : list of {{"label": ..., "terms": [...]}} if action=split, else []
  reasoning        : one sentence

action=inject  : terms clearly share a specific educational domain concept worth surfacing
action=split   : 2-3 coherent sub-groups exist within this cluster
action=discard : generic language, morphological noise, or no meaningful shared concept

Be conservative — discard ambiguous clusters. A useful injection token must be specific
enough that its presence changes what a TF-IDF topic means."""


def gate_cluster(cid, terms, retries=2):
    prompt = GATE_PROMPT.format(cid=cid, terms=terms[:20])
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_GATE,
                messages=[
                    {"role": "system", "content": GATE_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
            )
            text = resp.choices[0].message.content.strip()
            return json.loads(text)
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {"action": "discard", "primary_category": None,
                        "subcategory": None, "split_into": [],
                        "reasoning": f"API error: {e}"}


gate_results = []
for i, row in clusters.iterrows():
    result = gate_cluster(row.cluster_id, row.terms)
    result["cluster_id"] = row.cluster_id
    result["terms"]      = row.terms
    gate_results.append(result)
    action = result.get("action", "?")
    label  = result.get("primary_category", "")
    if (i % 10 == 0) or action != "discard":
        print(f"  C{row.cluster_id:03d} [{action:7s}] {label or result.get('reasoning','')[:60]}")

gate_df = pd.DataFrame(gate_results)
print(f"\nGate summary:")
print(gate_df["action"].value_counts().to_string())

  C004 [inject ] latin_american_geography
  C014 [inject ] nautical_transportation
  C035 [inject ] historical_sites_and_monuments
  C036 [inject ] radio_broadcasting
  C042 [inject ] criminal_justice
  C043 [split  ] 'ottoman' and 'ottomans' refer to a furniture item, while 't
  C052 [inject ] olympic_sports
  C067 [inject ] public_health
  C079 [inject ] geography
  C081 [inject ] language_acquisition
  C083 [inject ] south_asian_countries
  C086 [inject ] literary_instruction
  C094 [inject ] music_education
  C100 [discard] These terms are too generic and broad, describing abstract s
  C116 [inject ] new_york_city_geography
  C123 [inject ] hospitality_industry
  C129 [inject ] archery
  C132 [inject ] phonics_pronunciation
  C140 [inject ] military_history
  C148 [discard] These terms are primarily generic temperature/state words wi
  C149 [inject ] school_sanitation
  C153 [inject ] microbiology
  C196 [inject ] world_geography
  C199 [inject ] adolescent_development
  C205 [spli

In [5]:
# ── Build semantic_map.csv ────────────────────────────────────────────────────
# Expand inject + split results into a flat term → category mapping.

sem_rows = []

for _, row in gate_df.iterrows():
    if row.action == "inject" and row.primary_category:
        for term in row.terms:
            sem_rows.append({
                "term": term,
                "primary_category": row.primary_category,
                "subcategory": row.get("subcategory"),
                "source_cluster": row.cluster_id
            })

    elif row.action == "split" and row.get("split_into"):
        for sub in row["split_into"]:
            for term in sub.get("terms", []):
                sem_rows.append({
                    "term": term,
                    "primary_category": sub["label"],
                    "subcategory": None,
                    "source_cluster": row.cluster_id
                })

semantic_map = pd.DataFrame(sem_rows)
semantic_map.to_csv(OUT("enrichment", "semantic_map.csv"), index=False)

print(f"semantic_map.csv: {len(semantic_map):,} terms mapped")
print(f"Unique primary categories: {semantic_map.primary_category.nunique()}")
print("\nTop categories by term count:")
print(semantic_map.primary_category.value_counts().head(20).to_string())

semantic_map.csv: 125 terms mapped
Unique primary categories: 42

Top categories by term count:
primary_category
world_geography            5
aviation                   4
school_sanitation          4
hospitality_industry       4
east_asian_studies         4
criminal_justice           4
art_design                 3
psychology                 3
audio_technology           3
mobile_technology          3
ecology                    3
nautical_transportation    3
physical_education         3
behavior_management        3
reading_materials          3
forensic_science           3
genetics                   3
alaska_studies             3
dance_cheerleading         3
marketing_communication    3


In [6]:
# ── Review cell — inspect before proceeding ───────────────────────────────────
# Run this to audit the semantic_map. Edit semantic_map.csv manually if needed,
# then reload with: semantic_map = pd.read_csv(OUT("enrichment", "semantic_map.csv"))

print("=== Injected clusters by category ===")
for cat, grp in semantic_map.groupby("primary_category"):
    sub = f" / {grp.subcategory.dropna().unique()[0]}" if grp.subcategory.notna().any() else ""
    print(f"\n  {cat}{sub}  ({len(grp)} terms)")
    print(f"    {grp.term.tolist()[:15]}")

print("\n=== Discarded clusters (sample) ===")
discarded = gate_df[gate_df.action == "discard"]
for _, row in discarded.head(10).iterrows():
    print(f"  C{row.cluster_id:03d}: {row.terms[:8]} — {row.reasoning}")

=== Injected clusters by category ===

  adolescent_development / youth_age_group  (3 terms)
    ['adolescence', 'juvenile', 'teenage']

  alaska_studies / alaskan_geography  (3 terms)
    ['alaska', 'alaskan', 'anchorage']

  archery / archery_equipment  (3 terms)
    ['archery', 'arrow', 'bow']

  art_design / aesthetics  (3 terms)
    ['aesthetic', 'aesthetically', 'aesthetics']

  attitude_and_engagement_language  (2 terms)
    ['zeal', 'zest']

  audio_technology / sound_amplification  (3 terms)
    ['amp', 'amplification', 'amplifier']

  aviation / aerospace_aviation  (4 terms)
    ['aerospace', 'airplane', 'aviation', 'plane']

  behavior_management / aggressive_and_violent_behavior  (3 terms)
    ['aggression', 'aggressive', 'violent']

  country_turkey  (1 terms)
    ['turkey']

  criminal_justice / incarceration_systems  (4 terms)
    ['incarcerate', 'incarceration', 'jail', 'prison']

  dance_cheerleading  (3 terms)
    ['poise', 'pom', 'pompoms']

  east_asian_studies / as

---
## Pass B — Framing taxonomy discovery

Two steps:
1. Send a representative vocab sample to the LLM — it proposes a taxonomy of framing/tone/context categories specific to DonorsChoose essay language
2. Classify the full vocab against that taxonomy in batches

**You review and edit `framing_taxonomy.json` between steps 1 and 2.**

In [7]:
# ── Step B1: Load curated taxonomy ───────────────────────────────────────────
# The framing taxonomy has been developed through an external research and review
# process and is loaded directly from file. The original LLM-based discovery call
# that previously lived here has been removed so it cannot overwrite the curated
# taxonomy. To update the taxonomy, edit the JSON file directly and re-run from
# this cell onward.
#
# Expected location: OUTPUTS/enrichment/framing_taxonomy.json
# Copy your curated file there before running.

taxonomy_path = OUT("enrichment", "framing_taxonomy.json")
if not taxonomy_path.exists():
    raise FileNotFoundError(
        f"framing_taxonomy.json not found at {taxonomy_path}\n"
        "Copy your curated taxonomy file to OUTPUTS/enrichment/ before running."
    )

with open(taxonomy_path) as f:
    taxonomy = json.load(f)

nmf_cats      = [c for c in taxonomy if c.get('tier') == 'nmf']
analysis_cats = [c for c in taxonomy if c.get('tier') == 'analysis']

print(f"Loaded {len(taxonomy)} categories from {taxonomy_path}")
print(f"  NMF-tier:      {len(nmf_cats)}")
print(f"  Analysis-tier: {len(analysis_cats)}")
print()
for cat in taxonomy:
    tier = cat.get('tier', '?')
    cov  = cat.get('target_coverage_pct', '?')
    print(f"  [{tier:8s}] {cat['category']:50s} {cov}")


Loaded 63 categories from OUTPUTS/enrichment/framing_taxonomy.json
  NMF-tier:      15
  Analysis-tier: 48

  [nmf     ] episodic_disruption_recovery                       20-35%
  [nmf     ] chronic_scarcity                                   18-32%
  [nmf     ] barrier_removal                                    25-40%
  [nmf     ] catch_up_remediation                               18-30%
  [nmf     ] opportunity_expansion                              20-35%
  [nmf     ] capacity_first_dignity                             18-30%
  [nmf     ] regulation_safety_need                             18-32%
  [nmf     ] readiness_to_learn                                 20-35%
  [nmf     ] loss_avoidance                                     18-32%
  [nmf     ] urgent_now_anchoring                               15-25%
  [nmf     ] future_preparation                                 18-30%
  [nmf     ] household_basic_needs_spillover                    15-25%
  [nmf     ] inclusion_accommodation_fra

In [8]:
# ── REVIEW CHECKPOINT ─────────────────────────────────────────────────────────
# Edit OUTPUTS/enrichment/framing_taxonomy.json before running the next cell.
#
# Things to consider:
#   - Rename categories to match your mental model
#   - Merge two similar ones, or split one that's too broad
#   - Delete any that don't match what you want to track
#   - Add a category the LLM missed (e.g. 'weather_environment' if not present)
#
# When happy, run the next cell to reload and proceed to classification.

with open(OUT("enrichment", "framing_taxonomy.json")) as f:
    taxonomy = json.load(f)

print(f"Loaded {len(taxonomy)} categories:")
for cat in taxonomy:
    print(f"  {cat['category']:35s} {cat['description'][:70]}")

Loaded 63 categories:
  episodic_disruption_recovery        Terms that frame need as event-shaped: disruption, crisis, interruptio
  chronic_scarcity                    Terms that frame need as persistent, background under-resourcing rathe
  barrier_removal                     Terms that describe removing an obstacle blocking ordinary participati
  catch_up_remediation                Terms that frame the request as helping students recover lost ground, 
  opportunity_expansion               Terms that frame the request as opening a new possibility, pathway, ex
  capacity_first_dignity              Terms that portray students through capability, pride, dignity, voice,
  regulation_safety_need              Terms that frame the request around calm, regulation, sensory comfort,
  readiness_to_learn                  Terms that frame the request as establishing the physical, cognitive, 
  loss_avoidance                      Terms that persuade by emphasizing what will be prevented, avoided, 

In [9]:
# ── Step B2: Classify full vocab in batches ───────────────────────────────────
# Classify all vocab terms (not just rare — framing appears in head vocab too).
# Most terms return null. ~5-10% expected to match a category.

BATCH_SIZE = 150  # terms per API call — tune down if hitting token limits

# Build taxonomy reference string for the prompt
# Build richer taxonomy reference — includes counter_examples and coverage
# so the classifier calibrates strictness correctly for each category.
def _fmt_cat(cat):
    lines = [
        f"  [{cat.get('tier','?')}] {cat['category']} (target: {cat.get('target_coverage_pct','?')})",
        f"    Description: {cat['description']}",
        f"    Examples: {', '.join(cat.get('examples', [])[:8])}",
        f"    Counter-examples (must return null): {', '.join(cat.get('counter_examples', [])[:6])}",
    ]
    return '\n'.join(lines)

taxonomy_ref = "\n\n".join(_fmt_cat(cat) for cat in taxonomy)
valid_categories = {cat["category"] for cat in taxonomy}

CLASSIFY_SYSTEM = (
    "You classify vocabulary terms from DonorsChoose teacher essays into framing/tone categories. "
    "Respond ONLY with a JSON object mapping term to category (or null). No preamble."
)

CLASSIFY_PROMPT = """Framing categories:
{taxonomy}

Classify each term below. Return null if the term doesn't clearly fit a framing/tone category
(most subject matter words, nouns, and neutral verbs should be null).
Only classify when the framing signal is clear and consistent across different essay contexts.

Terms: {terms}

Return only: {{"term1": "category_or_null", "term2": "category_or_null", ...}}"""


def classify_batch(terms_batch, retries=2):
    prompt = CLASSIFY_PROMPT.format(
        taxonomy=taxonomy_ref,
        terms=terms_batch
    )
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_CLASSIFY,
                messages=[
                    {"role": "system", "content": CLASSIFY_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
            )
            text = resp.choices[0].message.content.strip()
            raw  = json.loads(text)
            return {
                k: v for k, v in raw.items()
                if v is None or v in valid_categories
            }
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                print(f"    Batch error: {e}")
                return {t: None for t in terms_batch}


all_terms   = vocab_df.index.tolist()
batches     = [all_terms[i:i+BATCH_SIZE] for i in range(0, len(all_terms), BATCH_SIZE)]
framing_raw = {}

print(f"Classifying {len(all_terms):,} terms in {len(batches)} batches...")
for i, batch in enumerate(batches):
    result = classify_batch(batch)
    framing_raw.update(result)
    if (i + 1) % 10 == 0:
        classified = sum(1 for v in framing_raw.values() if v is not None)
        print(f"  Batch {i+1}/{len(batches)} — {classified:,} terms classified so far")

# Build framing_map
framing_map = pd.DataFrame([
    {"term": term, "framing_category": cat}
    for term, cat in framing_raw.items() if cat is not None
])
framing_map.to_csv(OUT("enrichment", "framing_map.csv"), index=False)

print(f"\nframing_map.csv: {len(framing_map):,} terms classified")
print(f"Coverage: {len(framing_map)/len(all_terms):.1%} of vocab")
print("\nCategory distribution:")
print(framing_map.framing_category.value_counts().to_string())

Classifying 7,297 terms in 49 batches...
  Batch 10/49 — 691 terms classified so far
  Batch 20/49 — 1,292 terms classified so far
  Batch 30/49 — 1,959 terms classified so far
  Batch 40/49 — 2,620 terms classified so far

framing_map.csv: 3,140 terms classified
Coverage: 43.0% of vocab

Category distribution:
framing_category
joy_possibility_appeal                  161
capacity_first_dignity                  155
place_community_specific_context        148
household_basic_needs_spillover         136
expressive_creative_production          125
future_preparation                      119
readiness_to_learn                      118
causal_specificity                      105
safety_protection_language               95
catch_up_remediation                     87
collaborative_social_learning            85
regulation_safety_need                   83
durable_infrastructure                   74
sensory_detail                           73
inclusion_accommodation_framing          70
experienti

In [10]:
# ── Framing map review ────────────────────────────────────────────────────────
# Spot-check each category before injection.

framing_map = pd.read_csv(OUT("enrichment", "framing_map.csv"))

print("=== Framing category members ===")
for cat, grp in framing_map.groupby("framing_category"):
    terms = grp.term.tolist()
    # Show with doc_freq for context
    with_freq = [
        f"{t} ({int(vocab_df.loc[t, 'doc_freq']):,})"
        for t in terms[:20] if t in vocab_df.index
    ]
    print(f"\n  [{cat}]  ({len(terms)} terms)")
    print(f"    {with_freq}")

# Flag any suspicious classifications
print("\n=== Terms classified in multiple passes (should not happen) ===")
overlap = set(semantic_map.term) & set(framing_map.term)
if overlap:
    print(f"  {len(overlap)} terms appear in both maps: {list(overlap)[:10]}")
    print("  These will get BOTH injection tokens, which is fine.")
else:
    print("  None — clean separation between subject and framing maps.")

=== Framing category members ===

  [ambitious_programmatic_ask]  (17 terms)
    ['academy (5,585)', 'ambitious (2,885)', 'enterprise (114)', 'extensive (2,410)', 'groundbreaking (32)', 'initiative (7,097)', 'innovation (6,128)', 'interdisciplinary (1,152)', 'mega (80)', 'module (962)', 'monumental (393)', 'premier (53)', 'program (114,209)', 'programme (583)', 'redesign (838)', 'reinvent (304)', 'workshop (9,737)']

  [anecdote_scene_setting]  (23 terms)
    ['aloud (14,253)', 'alouds (6,636)', 'arrival (1,088)', 'assembly (2,017)', 'bell (2,857)', 'bustle (774)', 'cafeteria (2,364)', 'center (84,868)', 'chatter (257)', 'circle (9,811)', 'gymnasium (735)', 'hall (3,419)', 'hallway (5,086)', 'lunchroom (187)', 'lunchtime (938)', 'meeting (7,493)', 'morning (30,021)', 'playtime (897)', 'recess (14,236)', 'scenario (3,845)']

  [authentic_audience_real_world]  (45 terms)
    ['advertise (351)', 'advertisement (251)', 'advertising (452)', 'audience (3,550)', 'audition (398)', 'authentic (

---
## Pass C — Token injection

Adds `__cat__`, `__sub__`, and `__framing__` tokens to each project's token list.
Original tokens are preserved — injection is additive.

Token naming convention:
- Subject: `__cat_marine_biology__`  and  `__sub_ocean_ecosystems__`
- Framing: `__framing_urgency__`

The double-underscore prefix ensures injected tokens are visually distinct in any output
and can be filtered or analyzed separately.

In [23]:
# ── Build injection lookups ───────────────────────────────────────────────────

semantic_map  = pd.read_csv(OUT("enrichment", "semantic_map.csv"))
framing_map   = pd.read_csv(OUT("enrichment", "framing_map.csv"))

MAX_DOC_FREQ_FOR_INJECTION = 25_000

# Terms that bypass the ceiling because they are definitionally load-bearing
# for a specific category. Format: term -> framing_category_name.
# A term listed here injects its paired category even if doc_freq exceeds
# MAX_DOC_FREQ_FOR_INJECTION. Only applies to framing tokens, not cat/sub tokens.
CEILING_EXCEPTIONS = {

    # barrier_removal — 'access' is the single most central term in this category
    'access':          'barrier_removal',
    'block':           'barrier_removal',

    # catch_up_remediation — comprehension anchors literacy remediation framing
    'comprehension':   'catch_up_remediation',

    # chronic_scarcity — lack and low are the core persistent-deprivation vocabulary
    'lack':            'chronic_scarcity',
    'low':             'chronic_scarcity',

    # collaborative_social_learning — collaborate is definitionally central
    'collaborate':     'collaborative_social_learning',
    'collaboration':   'collaborative_social_learning',

    # durable_infrastructure — chairs and boards are the quintessential durable items
    'board':           'durable_infrastructure',
    'chair':           'durable_infrastructure',

    # empathy_invitation — heart is the emotional anchor
    'heart':           'empathy_invitation',

    # event_news_disaster_anchoring — covid and pandemic ARE this category
    'covid':           'event_news_disaster_anchoring',
    'pandemic':        'event_news_disaster_anchoring',
    'event':           'event_news_disaster_anchoring',

    # future_preparation — forward-orientation anchors
    'advance':         'future_preparation',
    'beyond':          'future_preparation',

    # gratitude_orientation — 'thank' is the anchor term; appreciate and gift follow
    'thank':           'gratitude_orientation',
    'appreciate':      'gratitude_orientation',
    'gift':            'gratitude_orientation',

    # home_extension_family_colearning — family is the only high-freq signal
    'family':          'home_extension_family_colearning',

    # humble_plea — hope and please are load-bearing plea language
    'hope':            'humble_plea',
    'please':          'humble_plea',

    # inquiry_exploration — curiosity vocabulary is definitional
    'curious':         'inquiry_exploration',
    'curiosity':       'inquiry_exploration',
    'discover':        'inquiry_exploration',

    # mandate_compliance — curriculum is the core compliance anchor
    'curriculum':      'mandate_compliance',

    # opportunity_expansion — expand and engineering are the key enrichment signals
    'expand':          'opportunity_expansion',
    'engineering':     'opportunity_expansion',

    # regulation_safety_need — calm is the single most central term
    'calm':            'regulation_safety_need',

    # representation_cultural_reflection — culture is definitional
    'culture':         'representation_cultural_reflection',

    # rural_scarcity_dispersion — rural is literally definitional
    'rural':           'rural_scarcity_dispersion',

    # student_choice_autonomy — all four are core choice vocabulary
    'choice':          'student_choice_autonomy',
    'choose':          'student_choice_autonomy',
    'independent':     'student_choice_autonomy',
    'independently':   'student_choice_autonomy',
    'option':          'student_choice_autonomy',

    # system_bundle_request — set and kit are the primary bundle nouns
    'kit':             'system_bundle_request',
    'set':             'system_bundle_request',

    # teacher_as_caregiver — care and encourage are load-bearing
    'care':            'teacher_as_caregiver',
    'encourage':       'teacher_as_caregiver',

    # underfunding_inequity_context — poverty and funding are load-bearing
    'poverty':         'underfunding_inequity_context',
    'funding':         'underfunding_inequity_context',
    'socioeconomic':   'underfunding_inequity_context',

    # urgent_now_anchoring — core urgency vocabulary
    'critical':        'urgent_now_anchoring',
    'now':             'urgent_now_anchoring',
}

inject_lookup = {}

for _, row in semantic_map.iterrows():
    if vocab_df.loc[row.term, 'doc_freq'] > MAX_DOC_FREQ_FOR_INJECTION:
        continue
    tokens = [f"__cat_{row.primary_category}__"]
    if pd.notna(row.get("subcategory")) and row.subcategory:
        tokens.append(f"__sub_{row.subcategory}__")
    inject_lookup.setdefault(row.term, []).extend(tokens)

for _, row in framing_map.iterrows():
    term = row.term
    cat  = row.framing_category
    # Bypass ceiling only for the specific paired category in CEILING_EXCEPTIONS
    if vocab_df.loc[term, 'doc_freq'] > MAX_DOC_FREQ_FOR_INJECTION:
        if CEILING_EXCEPTIONS.get(term) != cat:
            continue
    inject_lookup.setdefault(term, []).append(f"__framing_{cat}__")

inject_lookup = {t: list(dict.fromkeys(toks))
                 for t, toks in inject_lookup.items()}

In [24]:
# ── Estimate injected token doc_freq before writing ───────────────────────────
# Any injected token that won't survive min_doc_count filter is flagged here.

min_doc_count = CFG["sql"]["min_doc_count"]

# Count how many projects contain at least one term mapping to each injected token
injected_token_counts = {}
for tokens_list in df["tokens"]:
    seen = set()
    for t in tokens_list:
        if t in inject_lookup:
            for inj in inject_lookup[t]:
                seen.add(inj)
    for inj in seen:
        injected_token_counts[inj] = injected_token_counts.get(inj, 0) + 1

inj_freq_df = pd.DataFrame([
    {"injected_token": tok, "project_count": cnt}
    for tok, cnt in injected_token_counts.items()
]).sort_values("project_count", ascending=False)

will_survive = inj_freq_df[inj_freq_df.project_count >= min_doc_count]
will_drop    = inj_freq_df[inj_freq_df.project_count < min_doc_count]

inj_freq_df.to_csv(OUT("enrichment", "injected_token_freq.csv"), index=False)

print(f"Injected tokens that will survive min_doc_count={min_doc_count}: {len(will_survive)}")
print(f"Injected tokens that will be filtered out:                       {len(will_drop)}")
if len(will_drop):
    print(f"  Will drop: {will_drop.injected_token.tolist()}")
print("\nTop injected tokens by project coverage:")
print(will_survive.head(20).to_string(index=False))

Injected tokens that will survive min_doc_count=25: 141
Injected tokens that will be filtered out:                       0

Top injected tokens by project coverage:
                            injected_token  project_count
        __framing_capacity_first_dignity__         372630
        __framing_joy_possibility_appeal__         354270
            __framing_readiness_to_learn__         288666
            __framing_causal_specificity__         287038
       __framing_student_choice_autonomy__         277918
            __framing_future_preparation__         274120
 __framing_collaborative_social_learning__         265057
               __framing_barrier_removal__         249491
          __framing_teacher_as_caregiver__         232320
__framing_expressive_creative_production__         229525
        __framing_durable_infrastructure__         229493
         __framing_gratitude_orientation__         220529
          __framing_urgent_now_anchoring__         216135
              __framing

In [25]:
# ── Injection preview ─────────────────────────────────────────────────────────
# Sample 20 projects and show before/after token lists for sanity check.

sample = df.sample(20, random_state=42)
preview_rows = []
for _, row in sample.iterrows():
    added = []
    for t in row["tokens"]:
        if t in inject_lookup:
            added.extend(inject_lookup[t])
    added = list(dict.fromkeys(added))
    preview_rows.append({
        "project_id": row["project_id"],
        "original_tokens": row["tokens"][:15],
        "injected_tokens": added
    })

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(OUT("enrichment", "injection_preview.csv"), index=False)

print("Injection preview (20 projects):")
for _, r in preview_df[preview_df.injected_tokens.apply(len) > 0].head(10).iterrows():
    print(f"\n  project {r.project_id}")
    print(f"    original: {r.original_tokens}")
    print(f"    injected: {r.injected_tokens}")

Injection preview (20 projects):

  project 9353901
    original: ['excite' 'share' 'opportunity' 'empower' 'special' 'education' 'hand'
 'experience' 'operate' 'base' 'coffee' 'cart' 'initiative' 'teach'
 'essential']
    injected: ['__framing_portable_flexible_resource__', '__framing_ambitious_programmatic_ask__', '__framing_collaborative_social_learning__', '__framing_institutional_accountability_signals__', '__framing_teacher_as_caregiver__', '__framing_student_choice_autonomy__', '__framing_future_preparation__', '__framing_causal_specificity__', '__framing_barrier_removal__', '__framing_inclusion_accommodation_framing__', '__framing_inquiry_exploration__', '__framing_partnership_investment__', '__framing_gratitude_orientation__']

  project 6242560
    original: ['love' 'outside' 'perfect' 'teach' 'life' 'cycle' 'way' 'caterpillar'
 'turn' 'butterfly' 'watch' 'flower' 'seed' 'child' 'observe']
    injected: ['__framing_observation_testimony__', '__framing_system_bundle_request__'

In [31]:
# ── Write enriched parquet ────────────────────────────────────────────────────

def inject_tokens(token_list, lookup):
    extra = []
    for t in token_list:
        if t in lookup:
            extra.extend(lookup[t])
    if not extra:
        return token_list
    # Deduplicate injected tokens (original tokens may repeat — that's fine)
    extra_dedup = list(dict.fromkeys(extra))
    return token_list + extra_dedup


df_enriched = df.copy()
# Convert to plain Python lists first — parquet round-trips can produce
# numpy arrays which cause shape-mismatch errors in inject_tokens.
df_enriched["tokens"] = df_enriched["tokens"].apply(
    lambda ts: inject_tokens(list(ts), inject_lookup)
)

# Stats
orig_len = df["tokens"].apply(len)
new_len  = df_enriched["tokens"].apply(len)
enriched_projects = (new_len > orig_len).sum()

print(f"Projects with at least one injected token: {enriched_projects:,} "
      f"({enriched_projects/len(df):.1%})")
print(f"Mean tokens before injection: {orig_len.mean():.1f}")
print(f"Mean tokens after injection:  {new_len.mean():.1f}")
print(f"Max injected tokens on any project: {(new_len - orig_len).max()}")

out_path = ROOT / "OUTPUTS/prepared/05_enriched.parquet"
df_enriched.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")
print("\nTo use in 02_tfidf_analysis.ipynb, change the load line to:")
print('  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")')

Projects with at least one injected token: 895,521 (99.9%)
Mean tokens before injection: 62.7
Mean tokens after injection:  72.8
Max injected tokens on any project: 36

Saved → OUTPUTS/prepared/05_enriched.parquet

To use in 02_tfidf_analysis.ipynb, change the load line to:
  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")


---
## Quality checkpoint

Verify the enriched parquet is plausible before handing off to `02_tfidf_analysis`.

In [27]:
# ── Injected token prevalence across categories ───────────────────────────────

cat_tokens    = [t for t in injected_token_counts if t.startswith("__cat_")]
framing_tokens = [t for t in injected_token_counts if t.startswith("__framing_")]

print("=== Subject category token coverage ===")
for tok in sorted(cat_tokens,
                  key=lambda t: injected_token_counts[t], reverse=True):
    cnt = injected_token_counts[tok]
    pct = cnt / len(df)
    bar = "▓" * int(pct * 200)
    print(f"  {tok:45s} {cnt:6,}  {pct:.1%}  {bar}")

print("\n=== Framing token coverage ===")
for tok in sorted(framing_tokens,
                  key=lambda t: injected_token_counts[t], reverse=True):
    cnt = injected_token_counts[tok]
    pct = cnt / len(df)
    bar = "▓" * int(pct * 200)
    print(f"  {tok:45s} {cnt:6,}  {pct:.1%}  {bar}")

=== Subject category token coverage ===
  __cat_literary_instruction__                   2,101  0.2%  
  __cat_criminal_justice__                       1,864  0.2%  
  __cat_aviation__                               1,798  0.2%  
  __cat_radio_broadcasting__                     1,786  0.2%  
  __cat_east_asian_studies__                     1,774  0.2%  
  __cat_psychology__                             1,752  0.2%  
  __cat_adolescent_development__                 1,695  0.2%  
  __cat_phonics_pronunciation__                  1,557  0.2%  
  __cat_art_design__                             1,542  0.2%  
  __cat_music_education__                        1,463  0.2%  
  __cat_school_sanitation__                      1,460  0.2%  
  __cat_forensic_science__                       1,427  0.2%  
  __cat_ecology__                                1,409  0.2%  
  __cat_latin_american_geography__               1,405  0.2%  
  __cat_south_asian_countries__                  1,369  0.2%  
  __cat_attitud

In [28]:
# ── Cross-tab: framing tone × project category ────────────────────────────────
# Early look at the analysis you care about: does framing differ by subject?
# Full analysis lives in 02_tfidf_analysis but this is a fast sanity check.
#
# Vectorized approach — avoids iterrows() over 896K rows.

framing_tok_set = set(framing_tokens)

if not framing_tok_set:
    print("No framing tokens found — check framing_map.csv and that cell 18 ran successfully.")
else:
    # Build a binary presence matrix: one column per framing token
    short_names = {ft: ft.replace('__framing_', '').replace('__', '') for ft in framing_tok_set}

    rows = {}
    for ft in framing_tok_set:
        col = short_names[ft]
        rows[col] = df_enriched["tokens"].apply(lambda ts: ft in ts)

    presence_df = pd.DataFrame(rows, index=df_enriched.index)
    presence_df["project_category"] = df_enriched["project_category"].values

    # Group by project_category and compute mean presence per framing token
    pivot_pct = presence_df.groupby("project_category").mean()

    pivot_pct.to_csv(OUT("enrichment", "framing_by_category.csv"))
    print("Framing token prevalence by project category (% of projects):")
    print((pivot_pct * 100).round(1).to_string())
    print(f"\nSaved → {OUT('enrichment', 'framing_by_category.csv')}")


Framing token prevalence by project category (% of projects):
                                triage_repair  system_bundle_request  new_program_launch  anecdote_scene_setting  joy_possibility_appeal  event_news_disaster_anchoring  teacher_as_caregiver  place_community_specific_context  inquiry_exploration  urgent_now_anchoring  teacher_workaround_visibility  regulation_safety_need  causal_specificity  invisible_enabling_supports  partnership_investment  underfunding_inequity_context  safety_protection_language  mandate_compliance  representation_cultural_reflection  gratitude_orientation  seasonal_school_calendar_anchoring  student_owned_personal_use  deadline_window_pressure  standards_research_justification  future_preparation  maintenance_replacement  authentic_audience_real_world  catch_up_remediation  collaborative_social_learning  rural_scarcity_dispersion  sensory_detail  experiential_hands_on_learning  subgroup_specific_framing  ambitious_programmatic_ask  expressive_creative_p

In [30]:
orig_len = pd.read_parquet("OUTPUTS/prepared/05_filtered.parquet")["tokens"].apply(len)
new_len  = df_enriched["tokens"].apply(len)
injected = new_len - orig_len

print("Injection count by original token length bucket:")
buckets = pd.cut(orig_len, bins=[0,20,40,60,80,200], labels=["≤20","21-40","41-60","61-80","80+"])
print(injected.groupby(buckets).describe()[["mean","50%","75%","max"]].round(1))

Injection count by original token length bucket:
        mean   50%   75%   max
tokens                        
≤20      3.1   3.0   4.0  10.0
21-40    5.5   5.0   7.0  17.0
41-60    8.4   8.0  10.0  22.0
61-80   11.1  11.0  13.0  28.0
80+     15.3  15.0  18.0  36.0


/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_29011/3602167099.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(injected.groupby(buckets).describe()[["mean","50%","75%","max"]].round(1))
